# ERC-20: explanations

Section 4.3, Table 9 and Fig. 2. Explanations refer to the CA + CMC Balanced Random Forest of Table 8, trained on the selected features, and to its test partition.

* **SHAP**: path-dependent TreeSHAP, mean absolute value for the scam class.
* **CIU**: mean contextual importance and utility over 200 scam tokens of the test partition.
* **DEXiRE**: rules extracted from an MLP distilled from the forest; thresholds are reported in original units.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split

from bfx.modeling import SEED, ciu_means, distil_and_extract, harmonise, readable_rule

DATA = ROOT / "data" / "erc20"
RESULTS = ROOT / "results" / "erc20"
MODELS = ROOT / "models" / "erc20"
SELECTED = pd.read_csv(RESULTS / "selected_features.csv")["feature"].tolist()

SOURCES = {
    "CA": ("chainabuse_scam_tokens", 1),
    "TO": ("tokenscout_scam_tokens", 1),
    "CMC": ("cmc_licit_tokens", 0),
    "TS": ("tokenscout_licit_tokens", 0),
}


def load(name):
    file, _ = SOURCES[name]
    df = pd.read_csv(DATA / "features" / f"{file}.csv")
    address = df["token_file"].str.removesuffix(".csv").str.rsplit("_", n=1).str[-1].str.lower()
    return pd.concat([df, address.rename("address")], axis=1)


def scenario(fraud, licit):
    data = harmonise({fraud: load(fraud), licit: load(licit)}, fraud_sources=[fraud], key="address")
    X = data.drop(columns=["token_file", "label", "source_directory", "address", "source", "y"], errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    limit = np.finfo(np.float32).max  # tree models work in float32
    return X.clip(-limit, limit), data["y"]


X, y = scenario("CA", "CMC")
X = X[SELECTED]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
model = joblib.load(MODELS / "CA_CMC.joblib")

## SHAP

In [ ]:
values = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent").shap_values(X_test)
values = values[:, :, 1] if values.ndim == 3 else values
shap_rank = pd.Series(np.abs(values).mean(axis=0), index=SELECTED, name="mean_abs_shap").sort_values(ascending=False)

In [ ]:
shap.summary_plot(values, X_test, feature_names=SELECTED, max_display=15, show=False)
plt.savefig(RESULTS / "fig2_shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()

## CIU

In [ ]:
scam_test = X_test[y_test.to_numpy() == 1].sample(200, random_state=SEED)
ciu = ciu_means(model, X_train, scam_test)
print(f"mean CI = {ciu['CI'].mean():.3f}   mean CU = {ciu['CU'].mean():.3f}")

## Table 9

In [ ]:
table9 = pd.concat([shap_rank, ciu], axis=1).sort_values("mean_abs_shap", ascending=False)
table9.to_csv(RESULTS / "table9_shap_ciu.csv")
table9.head(8).round(3)

## DEXiRE

In [ ]:
rules, scaler, figures, rule_pred, covered = distil_and_extract(model, X_train.to_numpy(), X_test.to_numpy(), SELECTED)
rule_texts = [str(r) for r in rules.get_rules()]
y_cov = y_test.to_numpy()[covered]
figures["rule_accuracy"] = accuracy_score(y_cov, rule_pred[covered])
figures["rule_balanced_accuracy"] = balanced_accuracy_score(y_cov, rule_pred[covered])
_ = (RESULTS / "dexire.json").write_text(json.dumps(figures, indent=2, default=float))
_ = (RESULTS / "dexire_rules.txt").write_text("\n".join(readable_rule(t, scaler, SELECTED) for t in rule_texts))
pd.Series(figures).round(3)

The five shortest rules:

In [ ]:
for t in sorted(rule_texts, key=len)[:5]:
    print(readable_rule(t, scaler, SELECTED))